# LAPORAN PRAKTIKUM & TUGAS CODING TRANSFORMASI GEOMETRI 2D
**Mata Kuliah:** Grafika Komputer dan Pengolahan Citra  
**Pertemuan:** Minggu ke-3 / Pertemuan 2 (Representasi 2D dan Transformasi Dasar)  

---
## DAFTAR ISI
1. [Dasar Teori dan Rumus Matematis](#1-dasar-teori-dan-rumus-matematis)
2. [Modul Fungsi Transformasi 2D](#2-modul-fungsi-transformasi-2d)
3. [Tugas 1: Rotasi Persegi 90° terhadap Titik Pusat](#3-tugas-1-rotasi-persegi-90-terhadap-titik-pusat)
4. [Tugas 2: Simulator 2D Sederhana (Poligon 5 Vertex)](#4-tugas-2-simulator-2d-sederhana-poligon-5-vertex)
5. [Tugas 3: Perbandingan Urutan Transformasi (T→R→S vs S→R→T)](#5-tugas-3-perbandingan-urutan-transformasi)
6. [Kesimpulan](#6-kesimpulan)


## 1. DASAR TEORI DAN RUMUS MATEMATIS

### A. Koordinat Homogen 2D
Pada grafika komputer 2D, sebuah titik $(x, y)$ dinyatakan dalam koordinat kartesius. Untuk memungkinkan operasi translasi, rotasi, dan skala digabungkan dalam satu perkalian matriks linear tunggal, digunakan sistem **Koordinat Homogen** dengan menambahkan komponen ketiga $w=1$:
$$
P = \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}
$$

### B. Matriks Transformasi Dasar terhadap Origin $(0,0)$:
1. **Translasi ($t_x, t_y$):**
   $$
   T(t_x, t_y) = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}
   \implies x' = x + t_x, \quad y' = y + t_y
   $$

2. **Rotasi $\theta$ (Counter-Clockwise) terhadap Origin:**
   $$
   R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}
   \implies x' = x\cos\theta - y\sin\theta, \quad y' = x\sin\theta + y\cos\theta
   $$

3. **Scaling ($s_x, s_y$) terhadap Origin:**
   $$
   S(s_x, s_y) = \begin{bmatrix} s_x & 0 & 0 \\ 0 & s_y & 0 \\ 0 & 0 & 1 \end{bmatrix}
   \implies x' = x \cdot s_x, \quad y' = y \cdot s_y
   $$

### C. Transformasi terhadap Titik Pusat / Pivot Arbitrer $(p_x, p_y)$:
Rotasi atau skala di sekeliling titik acuan $(p_x, p_y)$ dilakukan melalui 3 tahap komposit:
1. Translasi titik ke origin: $T(-p_x, -p_y)$
2. Lakukan transformasi dasar (rotasi atau scaling) di origin
3. Translasi kembali ke posisi awal: $T(p_x, p_y)$

$$
M_{R, pivot} = T(p_x, p_y) \cdot R(\theta) \cdot T(-p_x, -p_y)
$$
$$
M_{S, pivot} = T(p_x, p_y) \cdot S(s_x, s_y) \cdot T(-p_x, -p_y)
$$


## 2. MODUL FUNGSI TRANSFORMASI 2D
Implementasi modul matriks homogen 3x3 dan fungsi utilitas transformasi menggunakan library `numpy` dan `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ==============================================================================
# MODUL FUNGSI TRANSFORMASI 2D (MATRIKS HOMOGEN 3x3)
# ==============================================================================

def to_homogeneous(points):
    """Konversi titik kartesius (N, 2) ke koordinat homogen (N, 3)"""
    pts = np.asarray(points, dtype=float)
    if pts.ndim == 1:
        return np.append(pts, 1.0)
    return np.hstack([pts, np.ones((pts.shape[0], 1), dtype=float)])

def from_homogeneous(points_h):
    """Konversi kembali koordinat homogen (N, 3) ke kartesius (N, 2)"""
    pts_h = np.asarray(points_h, dtype=float)
    if pts_h.ndim == 1:
        return pts_h[:2] / pts_h[2]
    return pts_h[:, :2] / pts_h[:, 2:3]

def matriks_translasi(tx, ty):
    """Matriks translasi 3x3: [ [1,0,tx], [0,1,ty], [0,0,1] ]"""
    return np.array([
        [1.0, 0.0, float(tx)],
        [0.0, 1.0, float(ty)],
        [0.0, 0.0, 1.0]
    ], dtype=float)

def matriks_rotasi(theta_deg, px=0.0, py=0.0):
    """Matriks rotasi 3x3 sebesar theta_deg terhadap titik pusat (px, py)"""
    rad = np.radians(theta_deg)
    cos_t, sin_t = np.cos(rad), np.sin(rad)
    r_origin = np.array([
        [cos_t, -sin_t, 0.0],
        [sin_t,  cos_t, 0.0],
        [0.0,    0.0,   1.0]
    ], dtype=float)
    if px == 0.0 and py == 0.0:
        return r_origin
    t_to = matriks_translasi(-px, -py)
    t_back = matriks_translasi(px, py)
    return t_back @ r_origin @ t_to

def matriks_scaling(sx, sy, px=0.0, py=0.0):
    """Matriks scaling 3x3 dengan faktor (sx, sy) terhadap titik pusat (px, py)"""
    s_origin = np.array([
        [float(sx), 0.0,       0.0],
        [0.0,       float(sy), 0.0],
        [0.0,       0.0,       1.0]
    ], dtype=float)
    if px == 0.0 and py == 0.0:
        return s_origin
    t_to = matriks_translasi(-px, -py)
    t_back = matriks_translasi(px, py)
    return t_back @ s_origin @ t_to

def transform(points, matrix):
    """Transformasikan titik-titik dengan matriks 3x3"""
    pts = np.asarray(points, dtype=float)
    is_single = (pts.ndim == 1)
    pts_h = to_homogeneous(pts)
    if is_single:
        pts_new_h = matrix @ pts_h
    else:
        pts_new_h = pts_h @ matrix.T
    return from_homogeneous(pts_new_h)

def hitung_centroid(points):
    """Menghitung titik pusat massa (centroid)"""
    return np.mean(points, axis=0)

def close_poly(pts):
    """Menyambungkan titik awal di ujung akhir poligon agar tertutup saat digambar"""
    return np.vstack([pts, pts[0]])

print("Modul transformasi 2D berhasil dimuat!")

## 3. TUGAS 1: ROTASI PERSEGI 90° TERHADAP TITIK PUSAT

### A. Deskripsi & Rumus
Diberikan persegi dengan vertex $A(0,0), B(1,0), C(1,1), D(0,1)$.
Titik pusat persegi (centroid):
$$
P = \left(\frac{0+1+1+0}{4}, \frac{0+0+1+1}{4}\right) = (0.5, 0.5)
$$
Rotasi $90^\circ$ CCW terhadap $P(0.5, 0.5)$ dirumuskan dengan:
$$
x' = (x - 0.5)\cos(90^\circ) - (y - 0.5)\sin(90^\circ) + 0.5 = 1 - y
$$
$$
y' = (x - 0.5)\sin(90^\circ) + (y - 0.5)\cos(90^\circ) + 0.5 = x
$$

In [ ]:
# Data Persegi
persegi_awal = np.array([
    [0.0, 0.0],  # A
    [1.0, 0.0],  # B
    [1.0, 1.0],  # C
    [0.0, 1.0]   # D
], dtype=float)
labels_t1 = ['A', 'B', 'C', 'D']

pusat_t1 = hitung_centroid(persegi_awal)
px1, py1 = pusat_t1[0], pusat_t1[1]

# Matriks Rotasi 90 derajat terhadap pusat (0.5, 0.5)
M_rot90 = matriks_rotasi(90.0, px=px1, py=py1)
persegi_rot = transform(persegi_awal, M_rot90)

# Cetak Tabel Koordinat
print(f"Titik Pusat Persegi: ({px1:.2f}, {py1:.2f})")
print("-" * 55)
print(f"{'Vertex':<8} | {'Koordinat Awal':<18} | {'Hasil Rotasi 90°':<18}")
print("-" * 55)
for i, lbl in enumerate(labels_t1):
    xa, ya = persegi_awal[i]
    xr, yr = persegi_rot[i]
    print(f"  {lbl:<6} | ({xa:4.1f}, {ya:4.1f})        | ({xr:4.1f}, {yr:4.1f})")
print("-" * 55)

In [ ]:
# Plot Visualisasi Tugas 1
fig, ax = plt.subplots(figsize=(8, 7), dpi=120)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
ax.grid(True, linestyle=':', alpha=0.7)

p_awal_c = close_poly(persegi_awal)
p_rot_c = close_poly(persegi_rot)

# Plot Awal dan Rotasi
ax.plot(p_awal_c[:, 0], p_awal_c[:, 1], 'o-', color='#1f77b4', linewidth=2.2, label='Persegi Awal', zorder=4)
ax.fill(p_awal_c[:, 0], p_awal_c[:, 1], color='#1f77b4', alpha=0.18, zorder=2)
ax.plot(p_rot_c[:, 0], p_rot_c[:, 1], 's--', color='#d62728', linewidth=2.0, label='Hasil Rotasi 90° (Pusat)', zorder=5)
ax.fill(p_rot_c[:, 0], p_rot_c[:, 1], color='#d62728', alpha=0.15, zorder=3)

# Plot Titik Pusat
ax.plot(px1, py1, marker='*', markersize=14, color='#2ca02c', markeredgecolor='black',
        linestyle='None', label=f'Pusat Rotasi ({px1:.1f}, {py1:.1f})', zorder=6)

# Anotasi & Panah Kurva Rotasi
for i, lbl in enumerate(labels_t1):
    xa, ya = persegi_awal[i]
    xr, yr = persegi_rot[i]
    ax.text(xa - 0.08, ya - 0.08, f"{lbl}", color='#1f77b4', fontsize=11, fontweight='bold')
    ax.text(xr + 0.04, yr + 0.04, f"{lbl}'", color='#d62728', fontsize=11, fontweight='bold')
    arc = patches.FancyArrowPatch((xa, ya), (xr, yr), connectionstyle="arc3,rad=0.35",
                                  arrowstyle="->,head_width=3,head_length=5",
                                  color='#9467bd', linestyle='-.', linewidth=1.2, alpha=0.8)
    ax.add_patch(arc)

ax.set_aspect('equal')
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_title("Tugas 1: Rotasi Persegi 90° terhadap Titik Pusat (0.5, 0.5)", fontsize=12, fontweight='bold')
ax.set_xlabel("Sumbu X")
ax.set_ylabel("Sumbu Y")
ax.legend(loc='upper left', framealpha=0.9)
plt.show()

### Analisis Tugas 1:
- Bentuk persegi setelah rotasi $90^\circ$ terhadap titik pusat $(0.5, 0.5)$ menempati wilayah spasial yang sama persis (bounding box invariant) karena simetri rotasi lipat-4 ($C_4$).
- Posisi vertex mengalami perpindahan siklis sebesar $90^\circ$ CCW: $A(0,0) \to B(1,0)$, $B(1,0) \to C(1,1)$, $C(1,1) \to D(0,1)$, dan $D(0,1) \to A(0,0)$.
- Apabila rotasi dilakukan terhadap origin $(0,0)$, seluruh persegi akan terlempar ke kuadran II $[(0,0), (0,1), (-1,1), (-1,0)]$. Hal ini membuktikan pentingnya translasi ke origin sebelum rotasi dan translasi balik ke pivot.

## 4. TUGAS 2: SIMULATOR 2D SEDERHANA (POLIGON 5 VERTEX)

### A. Deskripsi & Algoritma
Simulator menerima parameter:
1. Translasi: $(t_x, t_y)$
2. Sudut Rotasi: $\theta$ (derajat)
3. Skala: $(s_x, s_y)$
4. Titik Pusat / Pivot: $(p_x, p_y)$ (dapat menggunakan centroid poligon secara otomatis atau input koordinat sembarang)

Urutan Matriks Komposit:
$$
M_{total} = T(t_x, t_y) \cdot \left[ T(p_x, p_y) \cdot R(\theta) \cdot T(-p_x, -p_y) \right] \cdot \left[ T(p_x, p_y) \cdot S(s_x, s_y) \cdot T(-p_x, -p_y) \right]
$$

In [ ]:
# Definisi Poligon 5 Vertex Asimetris
poly_5 = np.array([
    [1.0, 1.0],  # V1
    [4.0, 1.0],  # V2
    [5.0, 3.0],  # V3
    [3.0, 5.0],  # V4
    [1.0, 3.5]   # V5
], dtype=float)

# Parameter Simulator
tx, ty = 4.0, 2.5
sudut = 45.0
sx, sy = 1.5, 1.2
centroid_p5 = hitung_centroid(poly_5)
px2, py2 = centroid_p5[0], centroid_p5[1]

# Matriks Komposit
m_t = matriks_translasi(tx, ty)
m_r_piv = matriks_rotasi(sudut, px=px2, py=py2)
m_s_piv = matriks_scaling(sx, sy, px=px2, py=py2)
m_total_sim = m_t @ m_r_piv @ m_s_piv

poly_trans = transform(poly_5, m_total_sim)
pivot_akhir = transform(centroid_p5, m_t)

# Tampilkan Tabel
print(f"Parameter: tx={tx}, ty={ty}, theta={sudut}°, sx={sx}, sy={sy}, Pivot=({px2:.2f}, {py2:.2f})")
print("-" * 60)
print(f"{'Vertex':<8} | {'Koordinat Asli':<20} | {'Hasil Transformasi':<20}")
print("-" * 60)
for i in range(len(poly_5)):
    va = poly_5[i]
    vh = poly_trans[i]
    print(f"  V{i+1:<5} | ({va[0]:5.2f}, {va[1]:5.2f})           | ({vh[0]:5.2f}, {vh[1]:5.2f})")
print("-" * 60)

In [ ]:
# Plot Visualisasi Simulator 2D
fig, ax = plt.subplots(figsize=(9, 8), dpi=120)
ax.axhline(0, color='#888888', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axvline(0, color='#888888', linestyle='--', linewidth=0.8, alpha=0.7)
ax.grid(True, which='both', color='#e0e0e0', linestyle=':', linewidth=0.9)

p_orig_c = close_poly(poly_5)
p_tr_c = close_poly(poly_trans)

# Poligon Asli & Hasil
ax.plot(p_orig_c[:, 0], p_orig_c[:, 1], 'o-', color='#1f77b4', linewidth=2.0, label='Poligon Asli (5 Vertex)', zorder=4)
ax.fill(p_orig_c[:, 0], p_orig_c[:, 1], color='#1f77b4', alpha=0.20, zorder=2)
ax.plot(p_tr_c[:, 0], p_tr_c[:, 1], 's-', color='#ff7f0e', linewidth=2.0, label='Hasil Transformasi (T+R+S)', zorder=5)
ax.fill(p_tr_c[:, 0], p_tr_c[:, 1], color='#ff7f0e', alpha=0.22, zorder=3)

# Pivot Point
ax.plot(px2, py2, marker='*', markersize=13, color='#2ca02c', label=f'Pivot Awal ({px2:.2f}, {py2:.2f})', zorder=6)
ax.plot(pivot_akhir[0], pivot_akhir[1], marker='*', markersize=13, color='#d62728', label=f'Pivot Akhir ({pivot_akhir[0]:.2f}, {pivot_akhir[1]:.2f})', zorder=6)
ax.annotate("", xy=(pivot_akhir[0], pivot_akhir[1]), xytext=(px2, py2),
            arrowprops=dict(arrowstyle="->", color='#2ca02c', linestyle='--', linewidth=1.5, alpha=0.8))

for i in range(len(poly_5)):
    ax.text(poly_5[i, 0] - 0.2, poly_5[i, 1] + 0.15, f"V{i+1}", color='#1f77b4', fontweight='bold')
    ax.text(poly_trans[i, 0] + 0.15, poly_trans[i, 1] + 0.15, f"V{i+1}'", color='#d35400', fontweight='bold')

ax.set_aspect('equal')
ax.set_title("Tugas 2: Simulator Transformasi Geometri 2D (5 Vertex)", fontsize=12, fontweight='bold')
ax.set_xlabel("Sumbu X")
ax.set_ylabel("Sumbu Y")
ax.legend(loc='upper left', framealpha=0.9)
plt.show()

## 5. TUGAS 3: PERBANDINGAN URUTAN TRANSFORMASI

### A. Pertanyaan & Analisis Aljabar Matriks
**Pertanyaan:** Bandingkan hasil Translasi $\to$ Rotasi $\to$ Scaling ($T \to R \to S$) dengan Scaling $\to$ Rotasi $\to$ Translasi ($S \to R \to T$) pada satu vertex. Apakah hasilnya sama?

**Analisis Matematis:**
- Matriks Komposit Urutan 1 ($T \to R \to S$):
  $$
  M_{TRS} = S \cdot R \cdot T = \begin{bmatrix} s_x\cos\theta & -s_x\sin\theta & s_x(t_x\cos\theta - t_y\sin\theta) \\ s_y\sin\theta & s_y\cos\theta & s_y(t_x\sin\theta + t_y\cos\theta) \\ 0 & 0 & 1 \end{bmatrix}
  $$
- Matriks Komposit Urutan 2 ($S \to R \to T$):
  $$
  M_{SRT} = T \cdot R \cdot S = \begin{bmatrix} s_x\cos\theta & -s_y\sin\theta & t_x \\ s_x\sin\theta & s_y\cos\theta & t_y \\ 0 & 0 & 1 \end{bmatrix}
  $$

Karena perkalian matriks **tidak bersifat komutatif** ($A \cdot B \neq B \cdot A$), kolom translasi dan elemen rotasi-skala pada kedua matriks berbeda secara fundamental!

In [ ]:
# Eksperimen pada satu vertex V(2.0, 3.0)
v_uji = np.array([2.0, 3.0], dtype=float)
tx3, ty3 = 4.0, 2.0
sudut3 = 90.0
sx3, sy3 = 2.0, 3.0

T = matriks_translasi(tx3, ty3)
R = matriks_rotasi(sudut3, px=0.0, py=0.0)
S = matriks_scaling(sx3, sy3, px=0.0, py=0.0)

# Urutan 1: T -> R -> S
M_TRS = S @ R @ T
v1_t = transform(v_uji, T)
v1_tr = transform(v1_t, R)
v1_trs = transform(v1_tr, S)

# Urutan 2: S -> R -> T
M_SRT = T @ R @ S
v2_s = transform(v_uji, S)
v2_sr = transform(v2_s, R)
v2_srt = transform(v2_sr, T)

print("=" * 65)
print("PERBANDINGAN LANGKAH PER LANGKAH PADA VERTEX V(2, 3):")
print("=" * 65)
print(f"Urutan 1 (T -> R -> S) : V(2,3) -> T(6,5) -> R(-5,6) -> S(-10, 18)")
print(f"Urutan 2 (S -> R -> T) : V(2,3) -> S(4,9) -> R(-9,4) -> T(-5, 6)")
print("-" * 65)
print(f"Hasil Akhir Urutan 1: V_TRS' = {v1_trs}")
print(f"Hasil Akhir Urutan 2: V_SRT' = {v2_srt}")
print(f"Jarak Euclidean Selisih = {np.linalg.norm(v1_trs - v2_srt):.4f} satuan")
print(f"Apakah M_TRS == M_SRT? {np.allclose(M_TRS, M_SRT)}")
print("=" * 65)

In [ ]:
# Plot Visualisasi Perbandingan Urutan Transformasi
poly_ref = np.array([[2.0, 3.0], [4.0, 3.0], [3.0, 5.0]])
poly_trs = transform(poly_ref, M_TRS)
poly_srt = transform(poly_ref, M_SRT)

fig, axs = plt.subplots(1, 2, figsize=(13, 6), dpi=120)
fig.suptitle("Perbandingan Urutan Transformasi: T→R→S vs S→R→T", fontsize=13, fontweight='bold')

# Subplot 1: T -> R -> S
axs[0].set_title("Urutan 1: T → R → S\nHasil: V' = (-10, 18)", fontsize=11, fontweight='bold', color='#1f77b4')
axs[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axs[0].axvline(0, color='gray', linestyle='--', linewidth=0.8)
axs[0].grid(True, linestyle=':', alpha=0.6)
s1 = np.array([v_uji, v1_t, v1_tr, v1_trs])
axs[0].plot(s1[:, 0], s1[:, 1], 'k:', linewidth=1.2)
axs[0].scatter(s1[:, 0], s1[:, 1], c=['black', 'green', 'orange', 'red'], s=60, zorder=5)
p_trs_c = close_poly(poly_trs)
p_orig_c = close_poly(poly_ref)
axs[0].plot(p_orig_c[:, 0], p_orig_c[:, 1], 'b-', label='Poligon Awal')
axs[0].fill(p_orig_c[:, 0], p_orig_c[:, 1], 'blue', alpha=0.15)
axs[0].plot(p_trs_c[:, 0], p_trs_c[:, 1], 'r--', label='Poligon Akhir (TRS)')
axs[0].fill(p_trs_c[:, 0], p_trs_c[:, 1], 'red', alpha=0.15)
axs[0].set_aspect('equal')
axs[0].legend(loc='lower left', fontsize=8.5)

# Subplot 2: S -> R -> T
axs[1].set_title("Urutan 2: S → R → T\nHasil: V' = (-5, 6)", fontsize=11, fontweight='bold', color='#d62728')
axs[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axs[1].axvline(0, color='gray', linestyle='--', linewidth=0.8)
axs[1].grid(True, linestyle=':', alpha=0.6)
s2 = np.array([v_uji, v2_s, v2_sr, v2_srt])
axs[1].plot(s2[:, 0], s2[:, 1], 'k:', linewidth=1.2)
axs[1].scatter(s2[:, 0], s2[:, 1], c=['black', 'purple', 'cyan', 'green'], s=60, zorder=5)
p_srt_c = close_poly(poly_srt)
axs[1].plot(p_orig_c[:, 0], p_orig_c[:, 1], 'b-', label='Poligon Awal')
axs[1].fill(p_orig_c[:, 0], p_orig_c[:, 1], 'blue', alpha=0.15)
axs[1].plot(p_srt_c[:, 0], p_srt_c[:, 1], 'g--', label='Poligon Akhir (SRT)')
axs[1].fill(p_srt_c[:, 0], p_srt_c[:, 1], 'green', alpha=0.15)
axs[1].set_aspect('equal')
axs[1].legend(loc='lower left', fontsize=8.5)

for ax in axs:
    ax.set_xlim(-12, 8)
    ax.set_ylim(-2, 20)
    ax.set_xlabel("Sumbu X")
    ax.set_ylabel("Sumbu Y")

plt.tight_layout()
plt.show()

## 6. KESIMPULAN

1. **Representasi Homogen 3x3:** Menggunakan koordinat homogen $[x, y, 1]^T$ memungkinkan operasi afinitas translasi digabungkan bersama rotasi dan skala menjadi perkalian matriks linear tunggal yang sangat efisien dalam pipeline rendering grafika komputer.
2. **Rotasi Terhadap Titik Pusat (Tugas 1):** Rotasi terhadap titik sembarang / pivot bukan origin wajib menggunakan skema translasi tiga langkah ($M = T(p) \cdot R(\theta) \cdot T(-p)$) agar objek berputar di tempat (in-place) dan tidak terlempar ke kuadran lain.
3. **Simulator 2D (Tugas 2):** Poligon sembarang dapat ditransformasikan secara fleksibel menggunakan matriks komposit terpadu dengan titik pivot yang disesuaikan secara dinamis.
4. **Sifat Non-Komutatif (Tugas 3):** Urutan transformasi sangat menentukan hasil akhir ($T \to R \to S \neq S \to R \to T$). Translasi yang diterapkan sebelum rotasi/skala akan menyebabkan pergeseran tersebut ikut diputar dan diperbesar/diperkecil oleh operasi berikutnya.